In [ ]:
# VS Code / local Jupyter environment check
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Run this notebook with the workspace folder as the current directory.")


# Local dataset setup (VS Code)

The training and test archives are already extracted in this workspace. No download or archive extraction is required.

Expected local folders:

```text
HV-AI-2025/HV-AI-2025/labeled_data/
HV-AI-2025/HV-AI-2025/unlabeled_data/
HV-AI-2025-Test/HV-AI-2025-Test/test_images/
```

Select a VS Code Jupyter kernel that has `torch`, `pandas`, `numpy`, and `Pillow`. A CUDA-enabled PyTorch kernel is strongly recommended.


In [ ]:
# Confirm that the local training folders are available.
training_root = PROJECT_ROOT / "HV-AI-2025" / "HV-AI-2025"
assert (training_root / "labeled_data" / "images").is_dir(), "Missing labeled image folder"
assert (training_root / "labeled_data" / "labeled_data.csv").is_file(), "Missing labels CSV"
assert (training_root / "unlabeled_data" / "images").is_dir(), "Missing unlabeled image folder"
print("Training data found:", training_root)


In [ ]:
# Confirm that the local test folder is available.
test_root = PROJECT_ROOT / "HV-AI-2025-Test" / "HV-AI-2025-Test" / "test_images"
assert test_root.is_dir(), "Missing test_images folder"
print("Test data found:", test_root)


# **Load/Preprocess **data****

In [ ]:
# Paths are resolved relative to the VS Code workspace containing this notebook.
project_root = Path.cwd()
LABELED_IMAGES_DIR = str(project_root / "HV-AI-2025" / "HV-AI-2025" / "labeled_data" / "images")
LABELS_CSV_PATH = str(project_root / "HV-AI-2025" / "HV-AI-2025" / "labeled_data" / "labeled_data.csv")
UNLABELED_IMAGES_DIR = str(project_root / "HV-AI-2025" / "HV-AI-2025" / "unlabeled_data" / "images")
TEST_IMAGES_DIR = str(project_root / "HV-AI-2025-Test" / "HV-AI-2025-Test" / "test_images")
OUTPUT_DIR = str(project_root / "outputs")


In [ ]:
from pathlib import Path
from collections import Counter
import copy
import csv
import random

import numpy as np
import pandas as pd
from PIL import Image, ImageFile, ImageEnhance, ImageOps
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

ImageFile.LOAD_TRUNCATED_IMAGES = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
if DEVICE.type == "cpu":
    print("CUDA is unavailable; using CPU. Training will be slower.")
assert all([LABELED_IMAGES_DIR, LABELS_CSV_PATH, UNLABELED_IMAGES_DIR]), \
    "Fill the labeled-images, labels-CSV, and unlabeled-images paths in the previous cell."

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
IMAGE_SIZE = 160
BATCH_SIZE = 128
SEED = 42
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True

seed_everything()
if DEVICE.type == "cpu":
    import os
    torch.set_num_threads(max(1, (os.cpu_count() or 4) - 2))

labels_df = pd.read_csv(LABELS_CSV_PATH)
path_column = next((c for c in labels_df.columns if c.lower() in {"path", "file", "filename"}), labels_df.columns[0])
label_column = next((c for c in labels_df.columns if c.lower() in {"label", "class"}), labels_df.columns[1])
labels_df[label_column] = labels_df[label_column].astype(str)
CLASS_NAMES = sorted(labels_df[label_column].unique().tolist())
assert len(CLASS_NAMES) == 10, f"Expected 10 classes, found {len(CLASS_NAMES)}: {CLASS_NAMES}"
class_to_id = {name: index for index, name in enumerate(CLASS_NAMES)}
id_to_class = {index: name for name, index in class_to_id.items()}

def resolve_labeled_path(value):
    candidate = Path(LABELED_IMAGES_DIR) / Path(str(value)).name
    if not candidate.is_file():
        candidate = Path(LABELED_IMAGES_DIR) / str(value)
    return candidate

labeled_paths = [resolve_labeled_path(value) for value in labels_df[path_column]]
labeled_labels = [class_to_id[value] for value in labels_df[label_column]]
unlabeled_paths = sorted(p for p in Path(UNLABELED_IMAGES_DIR).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
test_paths = (sorted(p for p in Path(TEST_IMAGES_DIR).rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
              if TEST_IMAGES_DIR else [])
missing = [path for path in labeled_paths if not path.is_file()]
assert not missing, f"Missing {len(missing)} labeled images; first missing path: {missing[0]}"
assert unlabeled_paths, "No unlabeled images were found. Check UNLABELED_IMAGES_DIR."
if TEST_IMAGES_DIR:
    assert test_paths, "No test images were found. Check TEST_IMAGES_DIR."
else:
    print("TEST_IMAGES_DIR is blank: training will run now and inference can run later.")

print("Device:", DEVICE)
print("Classes:", CLASS_NAMES)
print("Labeled class counts:", Counter(labels_df[label_column]))
print("Labeled / unlabeled / test:", len(labeled_paths), len(unlabeled_paths), len(test_paths))

class ImageDataset(Dataset):
    def __init__(self, paths, labels=None, transform=None):
        self.paths = list(paths)
        self.labels = None if labels is None else list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB")
        image = self.transform(image)
        if self.labels is None:
            return image, self.paths[index].name
        return image, self.labels[index]

def normalize_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    mean = torch.tensor(MEAN, dtype=tensor.dtype)[:, None, None]
    std = torch.tensor(STD, dtype=tensor.dtype)[:, None, None]
    return (tensor - mean) / std

class TrainTransform:
    def __call__(self, image):
        width, height = image.size
        scale = random.uniform(0.65, 1.0)
        aspect = random.uniform(0.85, 1.15)
        crop_width = min(width, max(1, int(width * (scale * aspect) ** 0.5)))
        crop_height = min(height, max(1, int(height * (scale / aspect) ** 0.5)))
        left = random.randint(0, max(0, width - crop_width))
        top = random.randint(0, max(0, height - crop_height))
        image = image.crop((left, top, left + crop_width, top + crop_height))
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        if random.random() < 0.5:
            image = ImageOps.mirror(image)
        if random.random() < 0.8:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.75, 1.25))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.75, 1.25))
            image = ImageEnhance.Color(image).enhance(random.uniform(0.8, 1.2))
        if random.random() < 0.05:
            image = ImageOps.grayscale(image).convert("RGB")
        tensor = normalize_image(image)
        if random.random() < 0.15:
            erase_height = random.randint(max(1, IMAGE_SIZE // 12), max(2, IMAGE_SIZE // 3))
            erase_width = random.randint(max(1, IMAGE_SIZE // 12), max(2, IMAGE_SIZE // 3))
            erase_top = random.randint(0, IMAGE_SIZE - erase_height)
            erase_left = random.randint(0, IMAGE_SIZE - erase_width)
            tensor[:, erase_top:erase_top + erase_height, erase_left:erase_left + erase_width] = 0
        return tensor

class EvalTransform:
    def __call__(self, image):
        image = ImageOps.fit(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            method=Image.Resampling.LANCZOS,
            centering=(0.5, 0.5),
        )
        return normalize_image(image)

train_transform = TrainTransform()
eval_transform = EvalTransform()

def stratified_split(labels, validation_fraction=0.15, seed=SEED):
    generator = random.Random(seed)
    train_indices, validation_indices = [], []
    for class_id in range(len(CLASS_NAMES)):
        indices = [i for i, label in enumerate(labels) if label == class_id]
        generator.shuffle(indices)
        validation_size = max(1, round(len(indices) * validation_fraction))
        validation_indices.extend(indices[:validation_size])
        train_indices.extend(indices[validation_size:])
    return train_indices, validation_indices

def make_loader(paths, labels, transform, balanced=False, shuffle=False):
    import os
    dataset = ImageDataset(paths, labels, transform)
    sampler = None
    if balanced and labels is not None:
        counts = Counter(labels)
        weights = [1.0 / counts[label] for label in labels]
        sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=0 if os.name == "nt" else 2,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=os.name != "nt",
    )

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, inputs):
        identity = inputs
        output = self.relu(self.bn1(self.conv1(inputs)))
        output = self.bn2(self.conv2(output))
        if self.downsample is not None:
            identity = self.downsample(inputs)
        return self.relu(output + identity)

class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(512, num_classes)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def _make_layer(self, out_channels, blocks, stride):
        layers = [BasicBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        layers.extend(BasicBlock(self.in_channels, out_channels) for _ in range(1, blocks))
        return nn.Sequential(*layers)

    def forward(self, inputs):
        output = self.maxpool(self.relu(self.bn1(self.conv1(inputs))))
        output = self.layer1(output)
        output = self.layer2(output)
        output = self.layer3(output)
        output = self.layer4(output)
        output = self.avgpool(output).flatten(1)
        return self.fc(self.dropout(output))

def build_resnet18():
    return ResNet18(len(CLASS_NAMES)).to(DEVICE)

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    per_class_correct, per_class_total = Counter(), Counter()
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.numel()
        for target, prediction in zip(labels.cpu().tolist(), predictions.cpu().tolist()):
            per_class_total[target] += 1
            per_class_correct[target] += int(target == prediction)
    accuracy = correct / total
    balanced_accuracy = np.mean([
        per_class_correct[class_id] / per_class_total[class_id]
        for class_id in range(len(CLASS_NAMES))
    ])
    return accuracy, float(balanced_accuracy)

def train_model(model, train_loader, validation_loader, epochs, learning_rate, checkpoint_path):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=learning_rate * 0.03)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    best_balanced_accuracy = -1.0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = samples = 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=USE_AMP):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
            samples += labels.size(0)
        scheduler.step()
        accuracy, balanced_accuracy = evaluate(model, validation_loader)
        print(f"Epoch {epoch:02d}/{epochs} | loss {running_loss/samples:.4f} | "
              f"val accuracy {accuracy:.4f} | balanced {balanced_accuracy:.4f}")
        if balanced_accuracy > best_balanced_accuracy:
            best_balanced_accuracy = balanced_accuracy
            best_weights = copy.deepcopy(model.state_dict())
            torch.save({"model": best_weights, "classes": CLASS_NAMES}, checkpoint_path)

    model.load_state_dict(best_weights)
    return model

@torch.inference_mode()
def predict_probabilities(model, paths):
    loader = make_loader(paths, None, eval_transform)
    model.eval()
    all_probabilities = []
    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        logits = (model(images) + model(torch.flip(images, dims=[3]))) / 2
        all_probabilities.append(logits.softmax(dim=1).cpu())
    return torch.cat(all_probabilities).numpy()

def write_predictions(paths, probabilities, output_path):
    predictions = probabilities.argmax(axis=1)
    frame = pd.DataFrame({
        "path": [path.name for path in paths],
        "predicted_label": [id_to_class[int(prediction)] for prediction in predictions],
    })
    frame.to_csv(output_path, index=False)
    print(f"Saved {len(frame)} predictions to {output_path}")
    return frame


# **Model initialization/Training**

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, paths, labels=None, transform=None):
        self.paths = list(paths)
        self.labels = None if labels is None else list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = image.convert("RGB")
        image = self.transform(image)
        if self.labels is None:
            return image, self.paths[index].name
        return image, self.labels[index]

def normalize_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1)
    mean = torch.tensor(MEAN, dtype=tensor.dtype)[:, None, None]
    std = torch.tensor(STD, dtype=tensor.dtype)[:, None, None]
    return (tensor - mean) / std

class TrainTransform:
    def __call__(self, image):
        width, height = image.size
        scale = random.uniform(0.65, 1.0)
        aspect = random.uniform(0.85, 1.15)
        crop_width = min(width, max(1, int(width * (scale * aspect) ** 0.5)))
        crop_height = min(height, max(1, int(height * (scale / aspect) ** 0.5)))
        left = random.randint(0, max(0, width - crop_width))
        top = random.randint(0, max(0, height - crop_height))
        image = image.crop((left, top, left + crop_width, top + crop_height))
        image = image.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.LANCZOS)
        if random.random() < 0.5:
            image = ImageOps.mirror(image)
        if random.random() < 0.8:
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.75, 1.25))
            image = ImageEnhance.Contrast(image).enhance(random.uniform(0.75, 1.25))
            image = ImageEnhance.Color(image).enhance(random.uniform(0.8, 1.2))
        if random.random() < 0.05:
            image = ImageOps.grayscale(image).convert("RGB")
        tensor = normalize_image(image)
        if random.random() < 0.15:
            erase_height = random.randint(max(1, IMAGE_SIZE // 12), max(2, IMAGE_SIZE // 3))
            erase_width = random.randint(max(1, IMAGE_SIZE // 12), max(2, IMAGE_SIZE // 3))
            erase_top = random.randint(0, IMAGE_SIZE - erase_height)
            erase_left = random.randint(0, IMAGE_SIZE - erase_width)
            tensor[:, erase_top:erase_top + erase_height, erase_left:erase_left + erase_width] = 0
        return tensor

class EvalTransform:
    def __call__(self, image):
        image = ImageOps.fit(
            image,
            (IMAGE_SIZE, IMAGE_SIZE),
            method=Image.Resampling.LANCZOS,
            centering=(0.5, 0.5),
        )
        return normalize_image(image)

train_transform = TrainTransform()
eval_transform = EvalTransform()

def stratified_split(labels, validation_fraction=0.15, seed=SEED):
    generator = random.Random(seed)
    train_indices, validation_indices = [], []
    for class_id in range(len(CLASS_NAMES)):
        indices = [i for i, label in enumerate(labels) if label == class_id]
        generator.shuffle(indices)
        validation_size = max(1, round(len(indices) * validation_fraction))
        validation_indices.extend(indices[:validation_size])
        train_indices.extend(indices[validation_size:])
    return train_indices, validation_indices

def make_loader(paths, labels, transform, balanced=False, shuffle=False):
    import os
    dataset = ImageDataset(paths, labels, transform)
    sampler = None
    if balanced and labels is not None:
        counts = Counter(labels)
        weights = [1.0 / counts[label] for label in labels]
        sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=0 if os.name == "nt" else 2,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=os.name != "nt",
    )

class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, inputs):
        identity = inputs
        output = self.relu(self.bn1(self.conv1(inputs)))
        output = self.bn2(self.conv2(output))
        if self.downsample is not None:
            identity = self.downsample(inputs)
        return self.relu(output + identity)

class ResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.25)
        self.fc = nn.Linear(512, num_classes)
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def _make_layer(self, out_channels, blocks, stride):
        layers = [BasicBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        layers.extend(BasicBlock(self.in_channels, out_channels) for _ in range(1, blocks))
        return nn.Sequential(*layers)

    def forward(self, inputs):
        output = self.maxpool(self.relu(self.bn1(self.conv1(inputs))))
        output = self.layer1(output)
        output = self.layer2(output)
        output = self.layer3(output)
        output = self.layer4(output)
        output = self.avgpool(output).flatten(1)
        return self.fc(self.dropout(output))

def build_resnet18():
    # Standard [2, 2, 2, 2] ResNet-18, randomly initialized from supplied data only.
    return ResNet18(len(CLASS_NAMES)).to(DEVICE)

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    per_class_correct, per_class_total = Counter(), Counter()
    for images, labels in loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.numel()
        for target, prediction in zip(labels.cpu().tolist(), predictions.cpu().tolist()):
            per_class_total[target] += 1
            per_class_correct[target] += int(target == prediction)
    accuracy = correct / total
    balanced_accuracy = np.mean([
        per_class_correct[class_id] / per_class_total[class_id]
        for class_id in range(len(CLASS_NAMES))
    ])
    return accuracy, float(balanced_accuracy)

def train_model(model, train_loader, validation_loader, epochs, learning_rate, checkpoint_path):
    criterion = nn.CrossEntropyLoss(label_smoothing=0.08)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=learning_rate * 0.03)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_balanced_accuracy = -1.0
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = samples = 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=DEVICE.type == "cuda"):
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 3.0)
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * labels.size(0)
            samples += labels.size(0)
        scheduler.step()
        accuracy, balanced_accuracy = evaluate(model, validation_loader)
        print(f"Epoch {epoch:02d}/{epochs} | loss {running_loss/samples:.4f} | "
              f"val accuracy {accuracy:.4f} | balanced {balanced_accuracy:.4f}")
        if balanced_accuracy > best_balanced_accuracy:
            best_balanced_accuracy = balanced_accuracy
            best_weights = copy.deepcopy(model.state_dict())
            torch.save({"model": best_weights, "classes": CLASS_NAMES}, checkpoint_path)

    model.load_state_dict(best_weights)
    return model

@torch.inference_mode()
def predict_probabilities(model, paths, use_tta=False, rotation_k=0, label="Inference"):
    loader = make_loader(paths, None, eval_transform)
    model.eval()
    all_probabilities = []
    total_batches = len(loader)
    for batch_index, (images, _) in enumerate(loader, start=1):
        images = images.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        if rotation_k:
            # torch.rot90 uses positive k for counter-clockwise rotation.
            images = torch.rot90(images, k=rotation_k, dims=(2, 3))
        logits = model(images)
        if use_tta:
            logits = (logits + model(torch.flip(images, dims=[3]))) / 2
        all_probabilities.append(logits.softmax(dim=1).cpu())
        if batch_index == 1 or batch_index % 20 == 0 or batch_index == total_batches:
            print(f"{label}: batch {batch_index}/{total_batches}")
    return torch.cat(all_probabilities).numpy()

def write_predictions(paths, probabilities, output_path):
    predictions = probabilities.argmax(axis=1)
    frame = pd.DataFrame({
        "path": [path.name for path in paths],
        "predicted_label": [id_to_class[int(prediction)] for prediction in predictions],
    })
    frame.to_csv(output_path, index=False)
    print(f"Saved {len(frame)} predictions to {output_path}")
    return frame


In [ ]:
OUTPUT_PATH = Path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

train_indices, validation_indices = stratified_split(labeled_labels)
train_paths = [labeled_paths[index] for index in train_indices]
train_labels = [labeled_labels[index] for index in train_indices]
validation_paths = [labeled_paths[index] for index in validation_indices]
validation_labels = [labeled_labels[index] for index in validation_indices]
train_loader = make_loader(train_paths, train_labels, train_transform, balanced=True)
validation_loader = make_loader(validation_paths, validation_labels, eval_transform)

# Phase 1: resume the completed labeled-only model when its checkpoint exists.
phase1_checkpoint = OUTPUT_PATH / "phase1_best.pt"
phase1_model = build_resnet18()
if phase1_checkpoint.is_file():
    saved = torch.load(phase1_checkpoint, map_location=DEVICE, weights_only=True)
    phase1_model.load_state_dict(saved["model"])
    print("Loaded existing Phase 1 checkpoint; Phase 1 training will not repeat.")
else:
    phase1_model = train_model(
        phase1_model, train_loader, validation_loader,
        epochs=30, learning_rate=3e-4, checkpoint_path=phase1_checkpoint,
    )

phase1_csv = OUTPUT_PATH / "phase1_predictions.csv"
rotation_marker = OUTPUT_PATH / "test_rotation_ccw90.done"
REBUILD_ROTATED_TEST_PREDICTIONS = not rotation_marker.is_file()
if not phase1_csv.is_file() or REBUILD_ROTATED_TEST_PREDICTIONS:
    phase1_probabilities = predict_probabilities(
        phase1_model, test_paths, rotation_k=1, label="Phase 1 rotated-test inference"
    )
    phase1_predictions = write_predictions(test_paths, phase1_probabilities, phase1_csv)
else:
    print("Phase 1 prediction CSV already exists; skipping repeated test inference.")

# Phase 2: cache unlabeled inference so interruptions never waste this work again.
pseudo_cache = OUTPUT_PATH / "unlabeled_probabilities.npy"
if pseudo_cache.is_file():
    unlabeled_probabilities = np.load(pseudo_cache)
    if len(unlabeled_probabilities) != len(unlabeled_paths):
        pseudo_cache.unlink()
        unlabeled_probabilities = predict_probabilities(
            phase1_model, unlabeled_paths, use_tta=False, label="Pseudo-label inference"
        )
        np.save(pseudo_cache, unlabeled_probabilities)
    else:
        print("Loaded cached unlabeled predictions.")
else:
    unlabeled_probabilities = predict_probabilities(
        phase1_model, unlabeled_paths, use_tta=False, label="Pseudo-label inference"
    )
    np.save(pseudo_cache, unlabeled_probabilities)

confidence = unlabeled_probabilities.max(axis=1)
pseudo_labels = unlabeled_probabilities.argmax(axis=1)
PSEUDO_THRESHOLD = 0.80
MAX_PSEUDO_PER_CLASS = 700
MIN_PSEUDO_PER_CLASS = 20
pseudo_paths_selected, pseudo_labels_selected = [], []

for class_id, class_name in enumerate(CLASS_NAMES):
    class_candidates = np.where(pseudo_labels == class_id)[0]
    class_candidates = class_candidates[np.argsort(-confidence[class_candidates])]
    confident = class_candidates[confidence[class_candidates] >= PSEUDO_THRESHOLD]
    # A small top-confidence fallback ensures every predicted class can contribute.
    selected = confident[:MAX_PSEUDO_PER_CLASS]
    if len(selected) < MIN_PSEUDO_PER_CLASS:
        selected = class_candidates[:min(MIN_PSEUDO_PER_CLASS, len(class_candidates))]
    pseudo_paths_selected.extend(unlabeled_paths[index] for index in selected)
    pseudo_labels_selected.extend([class_id] * len(selected))
    print(f"{class_name}: selected {len(selected)} pseudo-labels")

assert pseudo_paths_selected, "The model produced no usable pseudo-labels."

phase2_checkpoint = OUTPUT_PATH / "phase2_best.pt"
phase2_model = build_resnet18()
if phase2_checkpoint.is_file():
    saved = torch.load(phase2_checkpoint, map_location=DEVICE, weights_only=True)
    phase2_model.load_state_dict(saved["model"])
    print("Loaded existing Phase 2 checkpoint; Phase 2 training will not repeat.")
else:
    # Repeat true labels so pseudo-label noise cannot dominate training.
    phase2_paths = train_paths * 3 + pseudo_paths_selected
    phase2_labels = train_labels * 3 + pseudo_labels_selected
    phase2_loader = make_loader(phase2_paths, phase2_labels, train_transform, balanced=True)
    phase2_model.load_state_dict(phase1_model.state_dict())
    phase2_model = train_model(
        phase2_model, phase2_loader, validation_loader,
        epochs=5, learning_rate=7e-5, checkpoint_path=phase2_checkpoint,
    )

phase2_csv = OUTPUT_PATH / "phase2_predictions.csv"
if not phase2_csv.is_file() or REBUILD_ROTATED_TEST_PREDICTIONS:
    phase2_probabilities = predict_probabilities(
        phase2_model, test_paths, rotation_k=1, label="Phase 2 rotated-test inference"
    )
    phase2_predictions = write_predictions(test_paths, phase2_probabilities, phase2_csv)
    rotation_marker.write_text("Test tensors corrected 90 degrees counter-clockwise before inference.\n")
else:
    print("Rotation-corrected Phase 2 prediction CSV already exists.")


# **Model Inference**

**Test orientation correction:** visual inspection shows the supplied test images are rotated 90° clockwise. Inference rotates test tensors 90° counter-clockwise (`rotation_k=1`) before classification. Training and unlabeled images are not rotated by this correction.


In [ ]:
# Validate whichever phase outputs have completed; incomplete phases are reported, not crashed.
test_paths = sorted(
    path for path in Path(TEST_IMAGES_DIR).rglob("*")
    if path.suffix.lower() in IMAGE_EXTENSIONS
)
assert test_paths, "No test images were found. Check TEST_IMAGES_DIR."

for phase in (1, 2):
    checkpoint_path = OUTPUT_PATH / f"phase{phase}_best.pt"
    prediction_path = OUTPUT_PATH / f"phase{phase}_predictions.csv"
    if not checkpoint_path.is_file() or not prediction_path.is_file():
        print(f"Phase {phase} is incomplete. Rerun the training cell; it resumes automatically.")
        continue
    predictions = pd.read_csv(prediction_path)
    assert list(predictions.columns) == ["path", "predicted_label"]
    assert len(predictions) == len(test_paths)
    assert predictions["path"].is_unique
    assert set(predictions["path"]) == {path.name for path in test_paths}
    assert set(predictions["predicted_label"]).issubset(set(CLASS_NAMES))
    assert not predictions.isna().any().any()
    print(f"Phase {phase}: verified {len(predictions)} rows")
    display(predictions.head())


In [ ]:
# Show completed local submission files without failing on an interrupted phase.
for phase in (1, 2):
    prediction_path = OUTPUT_PATH / f"phase{phase}_predictions.csv"
    if prediction_path.is_file():
        print(prediction_path.resolve())
    else:
        print(f"Phase {phase} CSV is not ready; rerun the training cell to resume.")


# **Accuracy checks**

The supplied test package contains images only, so its true accuracy cannot be calculated locally unless HyperVerge provides a ground-truth CSV. The first cell reports honest held-out validation accuracy for each completed checkpoint. The second cell calculates exact test and per-class accuracy if a test-label CSV is later provided.


In [ ]:
@torch.inference_mode()
def detailed_accuracy_report(model, loader):
    model.eval()
    class_correct = Counter()
    class_total = Counter()
    total_correct = total_samples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        labels = labels.to(DEVICE, non_blocking=DEVICE.type == "cuda")
        predictions = model(images).argmax(dim=1)
        total_correct += (predictions == labels).sum().item()
        total_samples += labels.numel()
        for target, prediction in zip(labels.cpu().tolist(), predictions.cpu().tolist()):
            class_total[target] += 1
            class_correct[target] += int(target == prediction)

    per_class = pd.DataFrame([
        {
            "class": id_to_class[class_id],
            "correct": class_correct[class_id],
            "total": class_total[class_id],
            "accuracy": class_correct[class_id] / max(1, class_total[class_id]),
        }
        for class_id in range(len(CLASS_NAMES))
    ])
    overall = total_correct / max(1, total_samples)
    balanced = float(per_class["accuracy"].mean())
    return overall, balanced, per_class

for phase in (1, 2):
    checkpoint_path = OUTPUT_PATH / f"phase{phase}_best.pt"
    if not checkpoint_path.is_file():
        print(f"Phase {phase}: checkpoint not available yet.")
        continue
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
    accuracy_model = build_resnet18()
    accuracy_model.load_state_dict(checkpoint["model"])
    overall, balanced, per_class = detailed_accuracy_report(accuracy_model, validation_loader)
    print(f"Phase {phase} validation accuracy: {overall:.4%}")
    print(f"Phase {phase} balanced validation accuracy: {balanced:.4%}")
    display(per_class.style.format({"accuracy": "{:.2%}"}))
    del accuracy_model


In [ ]:
# Optional: enter a CSV path only if ground-truth test labels are provided.
# Required columns may be named path/filename/file and label/class.
TEST_LABELS_CSV = ""

if not TEST_LABELS_CSV:
    print("Test ground truth was not supplied, so test accuracy is unavailable locally.")
    print("The prediction CSVs can still be submitted to HyperVerge's evaluation server.")
else:
    truth = pd.read_csv(TEST_LABELS_CSV)
    truth_path_column = next(
        (column for column in truth.columns if column.lower() in {"path", "filename", "file"}),
        None,
    )
    truth_label_column = next(
        (column for column in truth.columns if column.lower() in {"label", "class"}),
        None,
    )
    assert truth_path_column and truth_label_column, "Ground-truth CSV needs path and label columns."
    truth = truth[[truth_path_column, truth_label_column]].copy()
    truth.columns = ["path", "true_label"]
    truth["path"] = truth["path"].map(lambda value: Path(str(value)).name)
    truth["true_label"] = truth["true_label"].astype(str)
    assert truth["path"].is_unique, "Ground-truth CSV contains duplicate paths."

    for phase in (1, 2):
        prediction_path = OUTPUT_PATH / f"phase{phase}_predictions.csv"
        if not prediction_path.is_file():
            print(f"Phase {phase}: prediction CSV not available yet.")
            continue
        predictions = pd.read_csv(prediction_path)
        predictions["predicted_label"] = predictions["predicted_label"].astype(str)
        merged = truth.merge(predictions, on="path", how="inner", validate="one_to_one")
        assert len(merged) == len(truth), "Some ground-truth paths are missing from predictions."
        merged["correct"] = merged["true_label"] == merged["predicted_label"]
        test_accuracy = float(merged["correct"].mean())
        per_class_accuracy = (
            merged.groupby("true_label", as_index=False)["correct"]
            .agg(["sum", "count", "mean"])
            .rename(columns={"sum": "correct", "count": "total", "mean": "accuracy"})
            .reset_index()
        )
        print(f"Phase {phase} test accuracy: {test_accuracy:.4%}")
        display(per_class_accuracy.style.format({"accuracy": "{:.2%}"}))


# **Helper Functions**

In [ ]:
import requests

def send_results_for_evaluation(name, csv_file, email):
    url = "http://43.205.49.236:5050/inference"
    files = {'file': open(csv_file, 'rb')}
    data = {'email': email, 'name':name}
    response = requests.post(url, files=files, data=data)
    return response.json()



# ***Test Inference***


This function is used to save the csv file and send it to the evaluation server.

Format of CSV file (Follow the header names strictly):

          path (str)               predicted_label(str)
    test_data/images/xx.jpg               class_1         
    test_data/images/yy.jpg               class_2         
             :                               :                          
             :                               :                          

Once the prediction file is saved as shown in the above format, you can send it to the evaluation server along with your email.

Caution: check your **email** before executing the cell.

IMPORTANT NOTE: "class_1", "class_2" are just placeholders. They are not actual class names. Replace them with actual class names.


In [ ]:
# Optional evaluation-server submission. Disabled by default to prevent accidental uploads.
SUBMIT_TO_EVALUATION_SERVER = False
NAME = ""  # Fill locally before evaluation; do not commit personal details.
EMAIL = ""  # Fill locally before evaluation; do not commit personal details.
CSV_TO_EVALUATE = str(OUTPUT_PATH / "phase2_predictions.csv")

if SUBMIT_TO_EVALUATION_SERVER:
    assert NAME and EMAIL, "Fill NAME and EMAIL before enabling submission."
    assert Path(CSV_TO_EVALUATE).is_file(), f"Missing CSV: {CSV_TO_EVALUATE}"
    print("Accuracy:", send_results_for_evaluation(NAME, CSV_TO_EVALUATE, EMAIL))
else:
    print("Evaluation submission is disabled. Set SUBMIT_TO_EVALUATION_SERVER=True when ready.")


In [ ]:
# Optional: after filling your details in the evaluation cell above, run it to submit results.
print("Submission files are available at:")
print(OUTPUT_PATH / "phase1_predictions.csv")
print(OUTPUT_PATH / "phase2_predictions.csv")
